In [ ]:
# Tải dataset kaggle_dataset.jsonl (cần upload lên session Kaggle)
# File jsonl này đã được sinh ra từ script generate_training_data.py
import json
from datasets import load_dataset

dataset = load_dataset("json", data_files="kaggle_dataset.jsonl", split="train")
print(dataset[0])

In [ ]:
!pip install -U git+https://github.com/huggingface/transformers.git --no-deps
!pip install -q -U peft trl bitsandbytes accelerate

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E4B-it"

# Setup 4-bit quantization for Kaggle
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments

# Format data to ChatML/Gemma prompt structure
def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

args = TrainingArguments(
    output_dir="gemma-4-E4B-it-finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    max_steps=100, # Chỉnh sửa số bước tùy ý
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    optim="paged_adamw_8bit"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=args,
    peft_config=lora_config,
    formatting_func=formatting_func,
    max_seq_length=512,
)

trainer.train()

In [ ]:
# Save LoRA adapter
trainer.model.save_pretrained("gemma-4-E4B-it-weather-lora")
tokenizer.save_pretrained("gemma-4-E4B-it-weather-lora")
print("Model saved successfully!")